In [2]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
import pandas as pd
import numpy as np, sqlite3
from src.config import DATA_RAW, DATA_PROCESSED
from src.db import load_table
import seaborn as sns
import yfinance as yf


DB = '/Users/admin/Desktop/carbon-portfolio-project-v2/data/carbon.db'
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [17]:
def load_raw(filename, **kwargs):
    """Load a raw CSV from data/raw folder."""
    return pd.read_csv(DATA_RAW / filename, low_memory=False, **kwargs)

In [35]:
with sqlite3.connect(DB, timeout = 30) as con:
    con.executescript((ROOT / 'sql' / 'schema.sql').read_text())

In [8]:
# Market factors:
YF = {'brent':'BZ=F',
      'wti':'CL=F',
      'natgas_hh':'NG=F',
      'stoxx600':'^STOXX',
      'vix':'^VIX',
      'us10y':'^TNX'}

# Start date:
START = '2012-01-01'

## Section 1:

## Section 2: yfinance market factors

In [12]:
def pull_yf(tickers, start=START):
    frames = []
    for name, tk in tickers.items():
        df = yf.download(tk, start=start, progress=False, auto_adjust=False)
        if df.empty:
            print(f"{name:12s} {tk:8s} NO DATA"); continue
        # collapse the (field, ticker) column MultiIndex -> just field
        df = df.copy()
        df.columns = df.columns.get_level_values(0) if isinstance(df.columns, pd.MultiIndex) else df.columns
        df.index.name = 'date'
        long = (df.reset_index()
                  .melt(id_vars='date', var_name='price_type', value_name='price')
                  .assign(factor=name))
        frames.append(long)
        d = df.dropna(how='all')
        print(f"{name:12s} {tk:8s} {d.index.min().date()} -> {d.index.max().date()}  n={len(d)}")
    out = pd.concat(frames, ignore_index=True)
    out['date'] = pd.to_datetime(out['date']).dt.tz_localize(None).dt.normalize()
    out['price_type'] = out['price_type'].str.lower().str.replace(' ', '_')   # 'Adj Close' -> 'adj_close'
    return out[['date', 'factor', 'price_type', 'price']]

In [13]:
mkt = pull_yf(YF)
print(mkt['price_type'].unique())
print(mkt.groupby('factor')['date'].agg(['min','max','count']).to_string())
print(mkt.head(3).to_string(index=False))

brent        BZ=F     2012-01-03 -> 2026-08-26  n=3666
wti          CL=F     2012-01-03 -> 2026-08-26  n=3683
natgas_hh    NG=F     2012-01-03 -> 2026-08-26  n=3684
stoxx600     ^STOXX   2012-01-03 -> 2026-08-26  n=3672
vix          ^VIX     2012-01-03 -> 2026-08-26  n=3684
us10y        ^TNX     2012-01-03 -> 2026-08-25  n=3681
['adj_close' 'close' 'high' 'low' 'open' 'volume']
                 min        max  count
factor                                
brent     2012-01-03 2026-08-26  21996
natgas_hh 2012-01-03 2026-08-26  22104
stoxx600  2012-01-03 2026-08-26  22032
us10y     2012-01-03 2026-08-25  22086
vix       2012-01-03 2026-08-26  22104
wti       2012-01-03 2026-08-26  22098
      date factor price_type      price
2012-01-03  brent  adj_close 112.129997
2012-01-04  brent  adj_close 113.699997
2012-01-05  brent  adj_close 112.739998


In [39]:
def load_eua(path):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date_utc']).dt.tz_localize(None).dt.normalize()
    t3 = df[df['indeks']=='t3pa'].groupby('date')['price'].mean()   # collapse within-series dups
    t2 = df[df['indeks']=='t2pa'].groupby('date')['price'].mean()
    eua = t3.combine_first(t2).sort_index()                         # t3 wins on overlap; t2 fills pre-t3 gap
    return eua.rename('eua_price').reset_index()

In [44]:
eua = load_eua(ROOT/'data/raw/prices_eua.csv')
eua_long = (eua.rename(columns={'eua_price':'price'})
               .assign(factor='eua', price_type='close')[['date','factor','price_type','price']])
eua_long['date'] = pd.to_datetime(eua_long['date']).dt.tz_localize(None).dt.normalize()

# combine with other market factors:
market_factors = (pd.concat([mkt, eua_long], ignore_index=True)
                    .drop_duplicates(['date','factor','price_type']))

# assign date as index:
mkt_out = market_factors.assign(date=lambda d: pd.to_datetime(d['date']).dt.strftime('%Y-%m-%d'))

assert mkt_out.duplicated(['date','factor','price_type']).sum() == 0
print("dups:", mkt_out.duplicated(['date','factor','price_type']).sum(), " rows:", len(mkt_out))

dups: 0  rows: 135300


In [48]:
print(market_factors.groupby('factor')['price_type'].nunique().to_string())   # eua=1, rest=6
print("total rows:", len(market_factors))

factor
brent        6
eua          1
natgas_hh    6
stoxx600     6
us10y        6
vix          6
wti          6
total rows: 135300


## Fama-French European factors

They come from empirical finance research (Eugene Fama & Kenneth French) showing that stock returns are largely explained by a handful of systematic "factors". These are common patterns that move groups of stocks together. Each factor is a return series: the daily return you'd earn holding a particular long-short portfolio.

Sever factors are pulled:
- mkt_rf — "market minus risk-free": the market's return above the risk-free rate. The classic market factor (CAPM beta). This is the broad equity risk premium.
- smb — "small minus big": return of small-cap stocks minus large-cap. Captures the historical small-firm premium (size factor).
- hml — "high minus low": high book-to-market (value stocks) minus low (growth stocks). The value factor.
- rmw — "robust minus weak": firms with robust profitability minus weak. The profitability/quality factor.
- cma — "conservative minus aggressive": firms that invest conservatively minus those investing aggressively. The investment factor.
- wml — "winners minus losers": recent winners minus losers. The momentum factor.
- rf — the risk-free rate itself (not a factor, the baseline you subtract to get excess returns).


In [29]:
import pandas_datareader.data as web


In [32]:
def pull_ff(start=START):
    ff5 = web.DataReader('Europe_5_Factors_Daily', 'famafrench', start=start)[0] / 100
    mom = web.DataReader('Europe_Mom_Factor_Daily', 'famafrench', start=start)[0] / 100
    mom.columns = ['WML']

    ff = ff5.join(mom, how='left')
    if isinstance(ff.index, pd.PeriodIndex):
        ff.index = ff.index.to_timestamp()          # Period -> Timestamp
    ff.index = pd.to_datetime(ff.index).tz_localize(None).normalize()
    ff.index.name = 'date'
    ff.columns = (ff.columns.str.strip().str.lower()
                    .str.replace('-', '_').str.replace(' ', ''))
    long = ff.reset_index().melt(id_vars='date', var_name='factor', value_name='value')
    return long.dropna(subset=['value'])

In [33]:
ff_factors = pull_ff()
print("factors:", sorted(ff_factors['factor'].unique()))

print(ff_factors.groupby('factor')['value'].agg(['mean','std','min','max']).to_string())
print(ff_factors.groupby('factor')['date'].agg(['min','max']).to_string())

/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader('Europe_5_Factors_Daily', 'famafrench', start=start)[0] / 100
/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:2: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  ff5 = web.DataReader('Europe_5_Factors_Daily', 'famafrench', start=start)[0] / 100
/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:2: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  ff5 = web.DataReader('Europe_5_Factors_Daily', 'famafrench', start=start)[0] / 100
/var/folders/bg/1y0v_kw15cnb9w8c866jb_28

factors: ['cma', 'hml', 'mkt_rf', 'rf', 'rmw', 'smb', 'wml']
            mean       std     min     max
factor                                    
cma     0.000002  0.003110 -0.0173  0.0131
hml     0.000101  0.005253 -0.0304  0.0438
mkt_rf  0.000333  0.010274 -0.1200  0.0854
rf      0.000061  0.000079  0.0000  0.0002
rmw     0.000055  0.003003 -0.0173  0.0267
smb    -0.000011  0.003981 -0.0333  0.0185
wml     0.000393  0.007046 -0.1087  0.0456
              min        max
factor                      
cma    2012-01-02 2026-06-30
hml    2012-01-02 2026-06-30
mkt_rf 2012-01-02 2026-06-30
rf     2012-01-02 2026-06-30
rmw    2012-01-02 2026-06-30
smb    2012-01-02 2026-06-30
wml    2012-01-02 2026-06-30


/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:3: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  mom = web.DataReader('Europe_Mom_Factor_Daily', 'famafrench', start=start)[0] / 100
/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:3: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  mom = web.DataReader('Europe_Mom_Factor_Daily', 'famafrench', start=start)[0] / 100
/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_7600/2067463892.py:8: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  ff.index = ff.index.to_timestamp()          # Period -> Timestamp


In [49]:
MKT_DDL = """
CREATE TABLE IF NOT EXISTS market_factors (
    date TEXT NOT NULL, factor TEXT NOT NULL, price_type TEXT NOT NULL, price REAL,
    PRIMARY KEY (date, factor, price_type));
CREATE INDEX IF NOT EXISTS ix_mkt_factor ON market_factors(factor);
"""
FF_DDL = """
CREATE TABLE IF NOT EXISTS ff_factors (
    date TEXT NOT NULL, factor TEXT NOT NULL, value REAL,
    PRIMARY KEY (date, factor));
CREATE INDEX IF NOT EXISTS ix_ff_factor ON ff_factors(factor);
"""

mkt_out = market_factors.assign(date=lambda d: pd.to_datetime(d['date']).dt.strftime('%Y-%m-%d'))
ff_out  = ff_factors.assign(date=lambda d: pd.to_datetime(d['date']).dt.strftime('%Y-%m-%d'))


In [50]:
with sqlite3.connect(DB, timeout=30) as con:
    con.executescript(MKT_DDL); con.executescript(FF_DDL); con.commit()
    load_table(con, mkt_out, 'market_factors')
    load_table(con, ff_out,  'ff_factors')
    for t in ['market_factors','ff_factors']:
        print(f"{t:16s}", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

  market_factors         135,300 rows
  ff_factors              26,474 rows
market_factors   135300
ff_factors       26474
